In [1]:
from utils.analysis.valuation import (
    CompanyAnalyzer,
    ComparisonAnalyzer,
    SectorAnalyzer,
    AnalysisWeights,
    CompanyReporter,
)
from utils.tools.config import INVESTMENT_PROFILES

In [2]:
# ANALYSIS CONFIGURATION
# Weights determine the importance of each category in the final score

# ============================================================================
# SELECT YOUR INVESTMENT PROFILE
# ============================================================================
# Options: 'balanced', 'value', 'growth', 'quality'
# - balanced: Balanced across all metrics (default)
# - value: Emphasis on valuation (35%) - for value investors
# - growth: Emphasis on growth (45%) - for growth investors
# - quality: Emphasis on profitability and health (35% each)
# ============================================================================

PROFILE_NAME = 'quality'  

# Load profile from config
if PROFILE_NAME and PROFILE_NAME in INVESTMENT_PROFILES:
    profile = INVESTMENT_PROFILES[PROFILE_NAME]
    custom_weights = AnalysisWeights(
        profitability=profile['profitability'],
        financial_health=profile['financial_health'],
        growth=profile['growth'],
        efficiency=profile['efficiency'],
        valuation=profile['valuation']
    )
    print(f"[OK] Selected profile: {PROFILE_NAME.upper()}")
    print(f"   {profile['description']}\n")
else:
    # Use default weights (balanced)
    custom_weights = None
    print("[OK] Using default weights (Balanced)\n")

# Initialize components
analyzer = CompanyAnalyzer(weights=custom_weights)
comparator = ComparisonAnalyzer(company_analyzer=analyzer)
sector_analyzer = SectorAnalyzer(company_analyzer=analyzer)
reporter = CompanyReporter()

# Show current configuration
print("Configured weights:")
print(f"   Profitability:      {analyzer.weights.profitability:.0%}")
print(f"   Financial Health:   {analyzer.weights.financial_health:.0%}")
print(f"   Growth:             {analyzer.weights.growth:.0%}")
print(f"   Efficiency:         {analyzer.weights.efficiency:.0%}")
print(f"   Valuation:          {analyzer.weights.valuation:.0%}")

[OK] Selected profile: QUALITY
   Quality-first approach (high ROIC, strong margins, low debt)

Configured weights:
   Profitability:      40%
   Financial Health:   35%
   Growth:             10%
   Efficiency:         10%
   Valuation:          5%


In [3]:
# INDIVIDUAL COMPANY ANALYSIS
# Fundamentally analyze a company using the configured profile

TICKER_TO_ANALYZE = "AAPL" 

print(f"Analyzing {TICKER_TO_ANALYZE} with {PROFILE_NAME.upper()} profile...\n")

result = analyzer.analyze(TICKER_TO_ANALYZE)

if result.get('success'):
    # Show full report
    print(reporter.render(result))
    
    # Additional information
    print("\n" + "="*65)
    print("INTERPRETATION:")
    print("="*65)
    conclusion = result['conclusion']
    print(f"Rating: {conclusion['overall']}")
    print(f"Total Score:  {result['scores']['total']:.1f}/100")
    
    # Show alerts if any
    all_alerts = []
    for category in ['profitability', 'financial_health', 'growth', 'efficiency', 'valuation']:
        alerts = result.get(category, {}).get('alerts', [])
        all_alerts.extend([(category, alert) for alert in alerts])
    
    if all_alerts:
        print(f"\n[!] {len(all_alerts)} ALERTS DETECTED:")
        for cat, alert in all_alerts[:5]:  # Show max 5
            print(f"   [{cat}] {alert}")
else:
    print(f"[ERROR] {result.get('error')}")

Analyzing AAPL with QUALITY profile...

ANALYSIS: Apple Inc. (AAPL)
Sector: Technology | Industry: Consumer Electronics
Country: United States | Currency: USD

─────────────────────────────────────────────────────────────────
SCORE SUMMARY
─────────────────────────────────────────────────────────────────
  🟡 Profitability      [███████████████░░░░░]   78.7
  🟠 Financial Health   [██████████░░░░░░░░░░]   53.6
  🟠 Growth             [███████████░░░░░░░░░]   59.2
  🟡 Efficiency         [████████████░░░░░░░░]   62.8
  🔴 Valuation          [████░░░░░░░░░░░░░░░░]   23.1
─────────────────────────────────────────────────────────────────
  🟡 TOTAL              [████████████░░░░░░░░]   63.6
  📋 Conclusion: FAIR

─────────────────────────────────────────────────────────────────
PROFITABILITY (Score: 78.7)
─────────────────────────────────────────────────────────────────
  ROIC:             23.5%        excellent
  ROE:              152.0%       excellent
  ROA:              24.4%        excellent

In [4]:
# MULTI-COMPANY COMPARISON
# Compare fundamentals of several companies in the same sector

TICKERS_TO_COMPARE = ["SYF", "CF", "AFL", "NEM"] 

print(f"Comparing {len(TICKERS_TO_COMPARE)} companies...\n")

comparison = comparator.compare(TICKERS_TO_COMPARE)

if comparison.get('success'):
    # Show summary table
    print("COMPARISON TABLE:\n")
    display(comparison['summary_df'])
    
    print(f"\n[OK] {comparison['valid_count']}/{len(TICKERS_TO_COMPARE)} companies analyzed successfully")
else:
    print(f"[ERROR] Comparison error: {comparison.get('error')}")

Comparing 4 companies...

COMPARISON TABLE:



,Ticker,Name,Sector,Profitability,Health,Growth,Efficiency,Valuation,Total,Conclusion
0,SYF,Synchrony Financial,Financial Services,100.000000,62.996765,41.083333,51.369856,91.843953,75.886385,GOOD
1,CF,"CF Industries Holdings, Inc.",Basic Materials,69.558286,80.578693,73.229167,31.917378,84.729999,70.777011,GOOD
2,AFL,Aflac Incorporated,Financial Services,66.407811,78.946391,11.416667,94.631408,71.119063,68.355122,GOOD
3,NEM,Newmont Corporation,Basic Materials,70.454516,82.078604,48.864583,34.766665,64.709437,68.507914,GOOD



[OK] 4/4 companies analyzed successfully


In [5]:
# COMPANY RANKING
# Sort companies by total score

if comparison.get('success'):
    print("\nRANKING BY TOTAL SCORE:\n")
    print(f"{'Rank':<6} {'Ticker':<8} {'Company':<32} {'Score':<8} {'Conclusion'}")
    print("-" * 70)
    
    for item in comparison['ranking']:
        name = item['name'][:30] if len(item['name']) > 30 else item['name']
        print(f"  #{item['rank']:<3} {item['ticker']:<8} {name:<32} "
            f"{item['score']:5.1f}  {item['conclusion']['overall']}")


RANKING BY TOTAL SCORE:

Rank   Ticker   Company                          Score    Conclusion
----------------------------------------------------------------------
  #1   SYF      Synchrony Financial               75.9  GOOD
  #2   CF       CF Industries Holdings, Inc.      70.8  GOOD
  #3   NEM      Newmont Corporation               68.5  GOOD
  #4   AFL      Aflac Incorporated                68.4  GOOD


In [6]:
# CATEGORY LEADERS
# Identify the best and worst in each dimension

if comparison.get('success'):
    category_names = {
        'profitability': 'Profitability',
        'financial_health': 'Financial Health',
        'growth': 'Growth',
        'efficiency': 'Efficiency',
        'valuation': 'Valuation'
    }
    
    print("\nCATEGORY LEADERS:\n")
    
    for cat, data in comparison['category_leaders'].items():
        name = category_names.get(cat, cat)
        best = data['best']
        worst = data['worst']
        
        print(f"{name}:")
        print(f"  [+] Best:   {best['ticker']:<6} - {best['name'][:25]:<25} (Score: {best['score']:5.1f})")
        print(f"  [-] Worst:  {worst['ticker']:<6} - {worst['name'][:25]:<25} (Score: {worst['score']:5.1f})")
        print()


CATEGORY LEADERS:

Profitability:
  [+] Best:   SYF    - Synchrony Financial       (Score: 100.0)
  [-] Worst:  AFL    - Aflac Incorporated        (Score:  66.4)

Financial Health:
  [+] Best:   NEM    - Newmont Corporation       (Score:  82.1)
  [-] Worst:  SYF    - Synchrony Financial       (Score:  63.0)

Growth:
  [+] Best:   CF     - CF Industries Holdings, I (Score:  73.2)
  [-] Worst:  AFL    - Aflac Incorporated        (Score:  11.4)

Efficiency:
  [+] Best:   AFL    - Aflac Incorporated        (Score:  94.6)
  [-] Worst:  CF     - CF Industries Holdings, I (Score:  31.9)

Valuation:
  [+] Best:   SYF    - Synchrony Financial       (Score:  91.8)
  [-] Worst:  NEM    - Newmont Corporation       (Score:  64.7)



In [7]:
# GROUP STATISTICS
# Statistical analysis of the compared companies

if comparison.get('success'):
    stats = comparison['group_stats']
    
    print("\nGROUP STATISTICS:\n")
    
    categories = [
        ('Total', 'total'),
        ('Profitability', 'profitability'),
        ('Financial Health', 'financial_health'),
        ('Growth', 'growth'),
        ('Efficiency', 'efficiency'),
        ('Valuation', 'valuation')
    ]
    
    print(f"{'Category':<20} {'Mean':<8} {'Median':<10} {'Min':<8} {'Max':<8} {'Std':<8}")
    print("-" * 75)
    
    for name, key in categories:
        if key in stats:
            s = stats[key]
            print(f"{name:<20} {s['mean']:6.1f}   {s['median']:7.1f}    "
                  f"{s['min']:6.1f}   {s['max']:6.1f}   {s['std']:6.1f}")


GROUP STATISTICS:

Category             Mean     Median     Min      Max      Std     
---------------------------------------------------------------------------
Total                  70.9      69.6      68.4     75.9      3.0
Profitability          76.6      70.0      66.4    100.0     13.6
Financial Health       76.2      79.8      63.0     82.1      7.7
Growth                 43.6      45.0      11.4     73.2     22.1
Efficiency             53.2      43.1      31.9     94.6     25.1
Valuation              78.1      77.9      64.7     91.8     10.7


In [8]:
# SECTOR PEER ANALYSIS
# Compare a specific company with sector competitors

TARGET_COMPANY = "AAPL"  # <- Target company
PEERS = ["MSFT", "GOOGL", "META", "AMZN"]  # <- Competitors

print(f"Analyzing {TARGET_COMPANY} vs sector peers...\n")

sector_result = sector_analyzer.analyze_vs_peers(
    ticker=TARGET_COMPANY,
    peers=PEERS
)

if sector_result.get('success'):
    print(f"COMPANY: {sector_result['company']['company_name']} ({TARGET_COMPANY})") 
    print(f"Sector: {sector_result['sector']}")
    print(f"Industry: {sector_result['industry']}")
    print(f"Peers analyzed: {sector_result['peer_count']}\n")
    
    # Relative position
    rel_pos = sector_result['relative_position']
    category_names = {
        'profitability': 'Profitability',
        'financial_health': 'Financial Health',
        'growth': 'Growth',
        'efficiency': 'Efficiency',
        'valuation': 'Valuation',
        'total': 'TOTAL'
    }
    
    print("RELATIVE POSITION:\n")
    
    for cat, data in rel_pos.items():
        if isinstance(data, dict) and 'company_score' in data:
            name = category_names.get(cat, cat.upper())
            diff = data['vs_avg']
            arrow = "^" if diff > 0 else "v" if diff < 0 else "="
            
            # Tag based on difference
            if diff > 5:
                tag = "[+]"
            elif diff < -5:
                tag = "[-]"
            else:
                tag = "[=]"
            
            print(f"{name}:")
            print(f"  Company score:   {data['company_score']:5.1f}")
            print(f"  Peer average:    {data['peer_avg']:5.1f}")
            print(f"  {tag} Difference:    {diff:+5.1f} {arrow}")
            print(f"  Ranking:         #{data['rank']} of {data['total_compared']}")
            print()
    
    # Percentile
    if 'percentiles' in sector_result and 'percentile' in sector_result['percentiles']:
        pct = sector_result['percentiles']
        print(f"PERCENTILE: {pct['percentile']:.1f}%")
        print(f"   Interpretation: {pct['interpretation']}")
        print(f"   Sample size: {pct['sample_size']} companies")
else:
    print(f"[ERROR] {sector_result.get('error')}")

Analyzing AAPL vs sector peers...

COMPANY: Apple Inc. (AAPL)
Sector: Technology
Industry: Consumer Electronics
Peers analyzed: 4

RELATIVE POSITION:

Profitability:
  Company score:    78.7
  Peer average:     70.0
  [+] Difference:     +8.7 ^
  Ranking:         #1 of 5

Financial Health:
  Company score:    53.6
  Peer average:     65.3
  [-] Difference:    -11.7 v
  Ranking:         #5 of 5

Growth:
  Company score:    59.2
  Peer average:     65.4
  [-] Difference:     -6.2 v
  Ranking:         #4 of 5

Efficiency:
  Company score:    62.8
  Peer average:     39.1
  [+] Difference:    +23.7 ^
  Ranking:         #1 of 5

Valuation:
  Company score:    23.1
  Peer average:     33.0
  [-] Difference:     -9.9 v
  Ranking:         #5 of 5

TOTAL:
  Company score:    63.6
  Peer average:     63.0
  [=] Difference:     +0.6 ^
  Ranking:         #2 of 5

PERCENTILE: 75.0%
   Interpretation: Above sector average
   Sample size: 4 companies


In [9]:
# COMPARE DIFFERENT PROFILES
# Analyze the same company with different investment strategies

TEST_TICKER = "AAPL"  # <- Change here

print(f"Analyzing {TEST_TICKER} with ALL available profiles:\n")
print(f"{'Profile':<20} {'Score':<8} {'Conclusion':<15} {'Description'}")
print("=" * 80)

for profile_name, profile_config in INVESTMENT_PROFILES.items():
    # Create analyzer with this profile
    weights = AnalysisWeights(
        profitability=profile_config['profitability'],
        financial_health=profile_config['financial_health'],
        growth=profile_config['growth'],
        efficiency=profile_config['efficiency'],
        valuation=profile_config['valuation']
    )
    
    profile_analyzer = CompanyAnalyzer(weights=weights)
    result = profile_analyzer.analyze(TEST_TICKER)
    
    if result.get('success'):
        score = result['scores']['total']
        conclusion = result['conclusion']['overall']
        desc = profile_config['description']
        
        print(f"{profile_name.upper():<20} {score:5.1f}    {conclusion:<15} {desc}")
    else:
        print(f"{profile_name.upper():<20} ERROR")

print("\nNote: The same asset can have different evaluations depending on your strategy")

Analyzing AAPL with ALL available profiles:

Profile              Score    Conclusion      Description
BALANCED              58.3    FAIR            Balanced approach across all dimensions
VALUE                 49.4    WEAK            Value investing focus (low P/E, high dividend yield)
GROWTH                56.2    FAIR            Growth investing focus (high revenue growth)
QUALITY               63.6    FAIR            Quality-first approach (high ROIC, strong margins, low debt)

Note: The same asset can have different evaluations depending on your strategy


In [10]:
# FULL SECTOR ANALYSIS
# Analyze a large group of companies from the same sector

SECTOR_NAME = "Technology"  # For reference
SECTOR_TICKERS = [
    "AAPL", "MSFT", "GOOGL", "META", "NVDA",
    "AMZN", "TSLA", "CRM", "ADBE", "INTC"
]

print(f"SECTOR ANALYSIS: {SECTOR_NAME}")
print(f"   Analyzing {len(SECTOR_TICKERS)} companies...\n")

sector_comparison = comparator.compare(SECTOR_TICKERS)

if sector_comparison.get('success'):
    ranking = sector_comparison['ranking']
    
    # Top 5
    print("TOP 5 IN SECTOR:\n")
    for item in ranking[:5]:
        print(f"  #{item['rank']} {item['ticker']:<6} Score: {item['score']:5.1f} - {item['conclusion']['overall']}")
    
    # Bottom 5
    print(f"\n[!] BOTTOM 5 IN SECTOR:\n")
    for item in ranking[-5:]:
        print(f"   {item['ticker']:<6} Score: {item['score']:5.1f} - {item['conclusion']['overall']}")
    
    # Statistics
    scores = [item['score'] for item in ranking]
    print(f"\nSECTOR STATISTICS:")
    print(f"   Companies analyzed:  {len(scores)}")
    print(f"   Average score:       {sum(scores)/len(scores):.1f}")
    print(f"   Median score:        {sorted(scores)[len(scores)//2]:.1f}")
    print(f"   Best score:          {max(scores):.1f}")
    print(f"   Worst score:         {min(scores):.1f}")
    print(f"   Standard deviation:  {(sum((x - sum(scores)/len(scores))**2 for x in scores) / len(scores))**0.5:.1f}")
else:
    print(f"[ERROR] {sector_comparison.get('error')}")

SECTOR ANALYSIS: Technology
   Analyzing 10 companies...

TOP 5 IN SECTOR:

  #1 NVDA   Score:  74.5 - GOOD
  #2 MSFT   Score:  65.6 - GOOD
  #3 AAPL   Score:  63.6 - FAIR
  #4 META   Score:  63.5 - FAIR
  #5 ADBE   Score:  63.1 - FAIR

[!] BOTTOM 5 IN SECTOR:

   GOOGL  Score:  62.9 - FAIR
   AMZN   Score:  59.9 - FAIR
   TSLA   Score:  51.7 - FAIR
   CRM    Score:  49.4 - WEAK
   INTC   Score:  21.9 - WEAK

SECTOR STATISTICS:
   Companies analyzed:  10
   Average score:       57.6
   Median score:        63.1
   Best score:          74.5
   Worst score:         21.9
   Standard deviation:  13.6


In [11]:
# AVAILABLE INVESTMENT PROFILES
# Quick reference of all configured profiles

print("AVAILABLE INVESTMENT PROFILES:\n")
print("=" * 80)

for i, (profile_name, config) in enumerate(INVESTMENT_PROFILES.items(), 1):
    print(f"\n{i}. {profile_name.upper()}")
    print(f"   {config['description']}")
    print(f"   Weights:")
    print(f"      - Profitability:      {config['profitability']:.0%}")
    print(f"      - Financial Health:   {config['financial_health']:.0%}")
    print(f"      - Growth:             {config['growth']:.0%}")
    print(f"      - Efficiency:         {config['efficiency']:.0%}")
    print(f"      - Valuation:          {config['valuation']:.0%}")

print("\n" + "=" * 80)
print("TIP: Change PROFILE_NAME in Cell 1 to select your strategy")
print("Profiles are defined in: utils/tools/config.py")

AVAILABLE INVESTMENT PROFILES:


1. BALANCED
   Balanced approach across all dimensions
   Weights:
      - Profitability:      30%
      - Financial Health:   30%
      - Growth:             15%
      - Efficiency:         10%
      - Valuation:          15%

2. VALUE
   Value investing focus (low P/E, high dividend yield)
   Weights:
      - Profitability:      20%
      - Financial Health:   25%
      - Growth:             10%
      - Efficiency:         10%
      - Valuation:          35%

3. GROWTH
   Growth investing focus (high revenue growth)
   Weights:
      - Profitability:      15%
      - Financial Health:   15%
      - Growth:             45%
      - Efficiency:         10%
      - Valuation:          15%

4. QUALITY
   Quality-first approach (high ROIC, strong margins, low debt)
   Weights:
      - Profitability:      40%
      - Financial Health:   35%
      - Growth:             10%
      - Efficiency:         10%
      - Valuation:          5%

TIP: Change PROFILE_NAM